# 25장 · 한 문의를 요구 추출과 답변 작성으로 나누기

승인을 받은 회원이 친구의 대리 수령과 케이블 반납을 함께 묻습니다. 규정과 문의를 읽고 직접 한 번 답해 본 뒤, 요구를 먼저 뽑는 방식으로 같은 문의에 답합니다. 어느 방식에서 어느 조건을 놓쳤는지 실제 두 답을 비교해 보세요.

## 모델에게 일을 넘기기 전에 써 보는 인수인계 메모

두 명이 원문을 2분 동안 읽습니다. 첫 사람은 원문을 덮은 뒤 “이 회원은 무엇을 이미 마쳤나? 담당자가 답해야 할 것은 무엇인가?”를 한 줄씩 씁니다. 둘째 사람은 메모와 R1~R4만 보고 답합니다. 두 사람이 다시 원문을 열어 비교하세요. 학생증이 필요하다는 말이 빠졌는지와 반납 문의 자체가 빠졌는지는 다른 문제입니다. 전자는 규정에서 찾을 수 있지만 후자는 메모만으로 복원하기 어렵습니다. 이 차이가 첫 호출의 지시를 결정합니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 먼저 사람이 쓸 답의 재료 찾기

Q1에서 이미 정해진 사실은 기종, 대여일, 승인 상태입니다. 질문은 두 개입니다. 친구가 대신 받아도 되는지, 직접 수령한다면 케이블을 언제 어디에 돌려줘야 하는지입니다. 대여 규정 R2와 R3을 읽고 이 네 종류의 정보를 표시하세요. “문의를 요약해”라고만 하면 질문이 짧아져도 다음 담당자가 답하는 데 필요한 날짜는 없어질 수 있습니다.

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
from pathlib import Path
import os
from openai import OpenAI

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
question = '6월 10일에 A형을 빌리기로 했고 승인 댓글도 받았어요. 수업이 늦게 끝나는데 친구가 대신 받아도 될까요? 제가 직접 받으면 케이블은 언제 어디로 돌려주면 되나요?'
print(rules)

R2는 수령 조건, R3은 반납 기한과 물건을 정합니다. R3의 “다음 날”을 실제 날짜로 쓰려면 수령 날짜가 남아 있어야 합니다. 모델이 어떤 답을 써야 할지 판단할 때 원문에서 보존할 사실과 해결할 요청을 나누어 보세요.

## 한 번에 답해 기준 답변 남기기

한 번 호출해 얻은 답을 기준으로 삼습니다. 두 단계로 나눴다는 이유만으로 더 나은 답이 되지는 않습니다. 첫 답도 이미 충분히 정확할 수 있습니다.

In [ ]:
direct = ask("대여 담당자로서 문의에 답하세요. 규정에서 근거가 되는 조항을 붙이세요.", rules + "\n회원 문의\n" + question)
print(direct)

직접 받은 답 옆에 대리 수령 가능 여부, 기기와 케이블, 반납 기한, 반납 장소를 적습니다. 날짜가 상대 표현이라면 언제부터 계산하는지도 읽으세요. 문장 길이는 이 네 조건의 정확성을 알려 주지 않습니다.

## 중간 결과를 다음 호출의 입력으로 쓰기

워크플로는 어떤 일을 어느 순서로 수행하고 결과를 누구에게 넘길지 정한 절차입니다. 여기서는 첫 번째 호출이 인수인계 메모를 쓰고 두 번째 호출이 그 메모와 규정을 받아 답합니다. 두 번째 호출에 첫 원문을 자동으로 전달하는 기능은 없습니다. 무엇을 넘길지 우리가 고릅니다.

In [ ]:
take_notes = "회원이 말한 사실과 해결할 요청을 나누세요. 기종·날짜·승인 여부와 요청을 빠짐없이 남기세요. 답은 아직 쓰지 마세요."
notes = ask(take_notes, question)
print(notes)

이번 출력에서 날짜와 승인 여부를 짚고, 서로 다른 물음에 각각 밑줄을 긋습니다. 사실과 요구가 분리돼 있으면 담당자가 답을 쓸 때 새로 추론할 부분을 줄일 수 있습니다.

In [ ]:
write_answer = "규정과 인수인계 메모로 회원에게 짧게 답하세요. 메모의 요청을 각각 처리하고 근거 조항을 붙이세요."
# 두 번째 호출에 보낼 내용을 보관해 문제가 생기면 다시 읽습니다.
second_input = rules + "\n인수인계 메모\n" + notes
draft = ask(write_answer, second_input)
print(draft)

기한이 틀렸다면 먼저 메모에 날짜가 남았는지 확인하세요. 날짜가 있었다면 규정 해석 또는 답안 작성에서 어긋난 것입니다. 단계를 나누면 앞 단계에서 이미 사라진 사실을 찾을 수 있습니다.

## 반납 문의가 빠진 메모를 일부러 넘기기

사람이 짧게 만든 아래 메모는 오류가 있는 시험 입력입니다. 두 번째 작성 지시는 그대로 두고 메모만 바꿉니다. 결과에 우연히 반납 설명이 붙을 수 있으므로 실제 답과 실제 입력을 둘 다 읽으세요.

In [ ]:
missing_note = "6월 10일 A형 승인 완료. 친구가 대신 받을 수 있는지 질문함."
print(ask(write_answer, rules + "\n인수인계 메모\n" + missing_note))

반납 기한이나 케이블이 빠졌다면 첫 단계의 누락이 이후 단계에 영향을 준 사례입니다. 반대로 모델이 반납까지 안내했더라도 누락된 질문을 메모에서 되살린 것은 아닙니다. 다른 문의에서도 같은 동작을 보장할 근거가 있는지 따져 보세요.

## 두 단계가 맞물린 메모와 답을 읽어 보기

**집필자가 쓴 메모 예시**　확인한 사실은 6월 10일 A형 대여 승인입니다. 요청 하나는 친구의 대리 수령 가능 여부, 다른 하나는 케이블 반납 시각과 장소입니다.

**이 메모를 받은 담당자의 답 예시**　친구가 대신 받을 수 없습니다. 6월 10일 오후 3~5시에 학생증을 지참해 가온관 1층 안내 데스크로 오세요. 기기와 케이블은 다음 날인 11일 오후 5시까지 같은 데스크에 반납하세요.

날짜 계산은 메모의 10일과 R3의 다음 날에서 나옵니다. 수령 시간과 장소는 R2에서 나옵니다. 자신의 실행 결과에서 각 조건이 어디에서 왔는지 짚어 보세요.

## 실습 · 처음 보는 문의를 인수인계하기

개인 실습 90분. 자료의 Q4를 읽고, 첫 호출 전에 보존할 날짜와 두 물음을 종이에 씁니다. 기준 답변, 요구 추출, 초안 작성을 직접 실행하세요. 원문→메모→답에서 처음 사라진 사실 하나를 찾아 첫 지시만 고칩니다. 누락이 없다면 “댓글만 쓰면 되는지”를 의도적으로 뺀 메모로 다시 시험하세요. 짝은 어떤 입력이 바뀌었는지 기록만 보고 찾아야 합니다.

In [ ]:
new_question = '6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.'
# 날짜·요청을 뽑은 뒤 두 번째 호출에 넘겨 보세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>new_question = &#x27;6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.&#x27;
direct_again = ask(&quot;회원의 문의에 규정으로 답하고 근거를 붙이세요.&quot;, rules + &quot;\n&quot; + new_question)
print(&quot;한 번의 답&quot;, direct_again)
for instruction in (take_notes, take_notes + &quot; 연장 요청 댓글을 쓰는 일과 승인이 끝나는 조건을 별도 질문으로 보존하세요.&quot;):
    memo = ask(instruction, new_question)
    received = rules + &quot;\n메모\n&quot; + memo
    print(&quot;메모&quot;, memo)
    print(&quot;메모로 작성한 답&quot;, ask(write_answer, received))</code></pre>
<p>대조 예시. 6월 10일 수령 → 원래 반납일은 11일. 원래 반납일 정오까지 원래 게시글에 요청하고 담당자 승인 댓글을 받아야 합니다. 표현이 달라도 네 조건의 근거를 R3·R4에서 제시할 수 있으면 됩니다.</p>
</details>

## 다음 작업에 가져갈 기록

기준 답변과 중간 메모, 수정한 지시를 저장하세요. 다음 장에서는 정보가 빠진 신청까지 이 작성 절차에 넣어도 되는지 결정합니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)